# Live 3D Hand Localization
This notebook connects to 3 webcams (indices 0, 1, and 2), detects hands in real-time using YOLOv8, and triangulates the 3D position.

In [ ]:
import sys
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# Add project root to path
sys.path.append(os.path.abspath('..'))
from scripts.localization.localize_hand import HandLocalizer

In [ ]:
# 1. Initialize Pipeline
config_path = '../configs/camera_params.yaml'

# Look for fine-tuned model, otherwise fallback to the base YOLO model
model_path = '../models/hand_detector/best.pt'
if not os.path.exists(model_path):
    print(f"Fine-tuned model not found at {model_path}. Falling back to default yolov8n.pt")
    model_path = 'yolov8n.pt'

localizer = HandLocalizer(config_path, model_path)

# 2. Initialize Webcams
cam_indices = [0, 1, 2] # Adjust these if your OS assigns different indices to the cameras
cams = [cv2.VideoCapture(idx) for idx in cam_indices]

# Check if all cameras opened successfully
for i, cam in enumerate(cams):
    if not cam.isOpened():
        print(f"Warning: Failed to open camera {cam_indices[i]}")

In [ ]:
# 3. Live 3D Plotting Loop
%matplotlib inline
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

cam_names = ['cam1', 'cam2', 'cam3']
cam_positions = [localizer.config['cameras'][cam_id]['position'] for cam_id in cam_names]

try:
    # Run for 100 frames as a demo. Change to `while True:` for infinite looping.
    for frame_idx in range(100):
        frames = []
        # Grab frames from all cameras
        for cam in cams:
            ret, frame = cam.read()
            if ret:
                frames.append(frame)
            else:
                frames.append(None)
                
        if any(f is None for f in frames):
            print("Failed to grab frames from one or more cameras. Skipping frame...")
            time.sleep(0.1)
            continue
            
        # Triangulate Hand Position
        estimated_3d_pos = localizer.localize_3d(frames)
        
        # Show 2D previews (Optional: Press 'q' in the CV2 window to quit early)
        preview = cv2.hconcat([cv2.resize(f, (320, 240)) for f in frames])
        cv2.imshow("Camera Feeds [Cam1 | Cam2 | Cam3]", preview)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("User exited.")
            break

        # --- Update 3D Plot ---
        ax.cla() # Clear previous points
        
        # Draw Cameras
        for name, pos in zip(cam_names, cam_positions):
            ax.scatter(*pos, c='blue', marker='s', s=100)
            ax.text(pos[0], pos[1], pos[2] + 15, name, color='darkblue', fontweight='bold')
            
        # Draw Origin
        ax.scatter(0, 0, 0, c='black', marker='x', s=100, label='Origin [0,0,0]')

        # Draw Hand
        if estimated_3d_pos is not None:
            ax.scatter(*estimated_3d_pos, c='red', marker='*', s=300, label='Hand')
            for pos in cam_positions:
                ax.plot([pos[0], estimated_3d_pos[0]], 
                        [pos[1], estimated_3d_pos[1]], 
                        [pos[2], estimated_3d_pos[2]], 'g--', alpha=0.5)
        else:
            ax.text2D(0.05, 0.95, "No hand detected in all 3 views", transform=ax.transAxes, color='red')

        # Fix axis limits so the plot doesn't bounce around
        ax.set_xlim([-200, 200])
        ax.set_ylim([-200, 200])
        ax.set_zlim([-150, 150])
        ax.set_xlabel('X (mm)')
        ax.set_ylabel('Y (mm)')
        ax.set_zlabel('Z (mm)')
        ax.set_title(f'Live 3D Hand Localization (Frame {frame_idx})')
        ax.legend(loc='upper right')

        # Refresh display
        clear_output(wait=True)
        display(fig)
        
except KeyboardInterrupt:
    print("\nStopped by user.")
finally:
    # Ensure hardware is released properly
    for cam in cams:
        cam.release()
    cv2.destroyAllWindows()
    print("Cameras released.")